## HDBSCAN
### Clustering

In [ ]:
#%reset -f 
#reset kernel
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score, adjusted_mutual_info_score
from sklearn.model_selection import train_test_split

from wordcloud import WordCloud

We could not execute HDBSCAN on the whole dataset due to memory constraint, therefore we used a subsample.
We considered using KMeans results

In [ ]:
subsample_mask = np.load("subsample_mask.npy")
data = pd.read_csv(f"without_stopwords_150.csv")
#we actually forgot to remove duplicates/empty rows after removing stopwords, so "we do it now" by creating a mask
mask=[]
for index, row in data.iterrows():
    try:
        if len(row['body'].strip()) == 0:
            mask.append(False)
        else:            
            mask.append(True)
    except:#if NaN
        mask.append(False)

data = data[mask]
data=data[subsample_mask]

#read npy file (pre-processing was done in 'unsupervised-pre-processing.ipynb')
X = np.load('X_reduced_150_lemmatizer_BoW.npy')
X = X[mask]
X=X[subsample_mask]

In [ ]:
try:
    data = np.load('hdbscan_clustering_results.npy')
except OSError:#If the file does not exist or cannot be read.
    hdbscan = HDBSCAN(
        min_cluster_size=6,
        min_samples=3,#When not set (default value = 6) we got 14911 outliers (too many so lower the value),
        cluster_selection_method="leaf",#Should produce more clusters than default
    )
    hdbscan_results=hdbscan.fit(X)
    #hdbscan.condensed_tree_.plot()

    data["cluster"]= hdbscan_results.labels_
    np.save('hdbscan_clustering_results.npy', data)

In [ ]:
#Plot the number of comments for each cluster, with outliers as a separate bar

plt.figure(figsize=(10, 5))
cluster_sizes = data["cluster"].value_counts()

plt.bar(["outliers"]+[str(i) for i in range(0,cluster_sizes.shape[0]-1)], cluster_sizes.values)
#plt.title(f"Number of Comments in Each Cluster (k={numberOfChoosenClusters})")
plt.xlabel("Cluster")
plt.ylabel("Number of Comments (log scale)")
#plt.xticks(cluster_sizes.index.insert(0, "outliers"))
#log scale on y axis
plt.yscale('log')
plt.bar_label(plt.bar(["outliers"]+[str(i) for i in range(0,cluster_sizes.shape[0]-1)], cluster_sizes.values))

plt.show()
print("Cluster sizes:", cluster_sizes)

data_no_outliers = data[data["cluster"]!=-1]#remove outliers
cluster_sizes = data_no_outliers["cluster"].value_counts()

In [ ]:
#plot ECDF of probability of being in a cluster for each cluster
# plt.figure(figsize=(8, 5))
# #we're not plotting outliers cause their probability of being in a cluster is 0
# for i in range(0, cluster_sizes.shape[0]):
#     sns.ecdfplot(data=probabilities[data["cluster"]==i], label=f"Cluster {i}")
# plt.xlabel("")
# plt.ylabel("Cumulative probability")
# plt.title("ECDF of probability of being in a cluster for each cluster")
# plt.legend()
# plt.show()


probabilities = hdbscan_results.probabilities_


plt.figure(figsize=(8, 5))

for i in range(0, cluster_sizes.shape[0]):
        sns.kdeplot(
        probabilities[data["cluster"]==i],
        label=f"Cluster {i}"
    )

plt.legend()
plt.title("Probability Density per Cluster")
plt.show()

In [ ]:
user_clusters = data_no_outliers.groupby('author')['cluster'].nunique().reset_index()#grouping by author
#user clusters is now a series with author and number of clusters
user_clusters.columns = ['author', 'num_clusters']#turn it into a DF

user_comments = data.groupby('author')['body'].nunique().reset_index()
user_comments.columns = ['author', 'num_comments']

# # Filter user_clusters to only those authors who have multiple comments
# authors_with_multiple_comments = data_no_outliers.groupby('author')['body'].nunique()
# authors_with_multiple_comments = authors_with_multiple_comments[authors_with_multiple_comments > 1].index
# user_clusters = user_clusters[user_clusters['author'].isin(authors_with_multiple_comments)]

plt.figure(figsize=(8, 5))
sns.ecdfplot(data=user_clusters, x='num_clusters')
sns.ecdfplot(data=user_comments, x='num_comments')
plt.xlabel('Clusters')
plt.ylabel('ECDF')
plt.title('ECDF of number of clusters per user and number of comments per user')
plt.grid(True)
x_min = 1
x_max = user_clusters['num_clusters'].max()+1
plt.xlim(0, x_max)
plt.xticks(range(0, x_max))
plt.legend(['Number of Clustenp.save('kmeans_cluster_labels.npy', data['cluster'].values)rs per User', 'Number of Comments per User'])

plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X[:, 0], X[:, 1], c=data['cluster'], cmap='tab10', alpha=0.6)
plt.xlabel('Truncated SVD Component 1')
plt.ylabel('Truncated SVD 2')
plt.title('SVD: First Two Components Colored by Cluster')
plt.colorbar(scatter, label='Cluster')
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X[:, 0], X[:, 1], c=data_no_outliers['cluster'], cmap='tab10', alpha=0.6)
plt.xlabel('Truncated SVD Component 1')
plt.ylabel('Truncated SVD 2')
plt.title('SVD: First Two Components Colored by Cluster without outliers')
plt.colorbar(scatter, label='Cluster')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X[:, 2], X[:, 3], c=data['cluster'], cmap='tab10', alpha=0.6)
plt.xlabel('Truncated SVD Component 3')
plt.ylabel('Truncated SVD 4')
plt.title('SVD: Third and Fourth Components Colored by Cluster')
plt.colorbar(scatter, label='Cluster')
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X[:, 2], X[:, 3], c=data_no_outliers['cluster'], cmap='tab10', alpha=0.6)
plt.xlabel('Truncated SVD Component 3')
plt.ylabel('Truncated SVD 4')
plt.title('SVD: Third and Fourth Components Colored by Cluster without outliers')
plt.colorbar(scatter, label='Cluster')
plt.grid(True)
plt.show()

In [ ]:
#Group by cluster and concatenate all comments in each cluster, then compute TF-IDF to understand important words in each cluster
cluster_documents=data_no_outliers[['body','cluster']].groupby('cluster').agg(' '.join).reset_index()
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(cluster_documents['body'])
terms = tfidf_vectorizer.get_feature_names_out()

#List the first 3 largest clusters
clusters_to_plot = data_no_outliers['cluster'].value_counts().nlargest(3).index
#add two clusters from the smallest
clusters_to_plot = clusters_to_plot.append(data_no_outliers['cluster'].value_counts().nsmallest(2).index)

#For each cluster, get the top 50 words by TF-IDF score and plot a word cloud
for cluster_num in clusters_to_plot:
    tfidf_scores = tfidf_matrix[cluster_num].toarray().flatten()
    top_indices = np.argsort(tfidf_scores)[-50:]  # Get indices of top 50 words
    top_terms = {terms[i]: tfidf_scores[i] for i in top_indices}

    wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(top_terms)

    print(f"cluster {cluster_num} has {cluster_sizes[cluster_num]} comments")
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud for Cluster {cluster_num}')
    plt.show()
    # if cluster_sizes[cluster_num]<10:
    #     for comment in data_no_outliers[data_no_outliers['cluster'] == cluster_num]['body'].values:
    #         print(comment)


In [ ]:
subreddit_cluster = data.groupby('subreddit')['cluster'].nunique().reset_index()#grouping by subreddit
#subreddit clusters is now a series with subreddit and number of clusters
subreddit_cluster.columns = ['subreddit', 'num_clusters']#turn it into a DF

# Filter subreddit_cluster to only those subreddits that have multiple comments
subreddits_with_multiple_comments = data.groupby('subreddit')['body'].nunique()
subreddits_with_multiple_comments = subreddits_with_multiple_comments[subreddits_with_multiple_comments > 1].index
subreddit_cluster = subreddit_cluster[subreddit_cluster['subreddit'].isin(subreddits_with_multiple_comments)]

plt.figure(figsize=(8, 5))
sns.ecdfplot(data=subreddit_cluster, x='num_clusters')
plt.xlabel('Number of Clusters Assigned to Subreddit')
plt.ylabel('ECDF')
plt.grid(True)
plt.show()

In [ ]:
#List the first 3 largest clusters
clusters_to_plot = data_no_outliers['cluster'].value_counts().nlargest(3).index
#add two clusters from the smallest
clusters_to_plot = clusters_to_plot.append(data_no_outliers['cluster'].value_counts().nsmallest(2).index)

#For each cluster, get the top 50 words by TF-IDF score and plot a word cloud
for cluster_num in clusters_to_plot:
    cluster_sizes = data_no_outliers['cluster'].value_counts()
    subredditsNames = " ".join(data_no_outliers[data_no_outliers['cluster'] == cluster_num]['subreddit'].values)
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(subredditsNames)

    print(f"Cluster {cluster_num} has {cluster_sizes[cluster_num]} comments and {data_no_outliers[data_no_outliers['cluster'] == cluster_num]['subreddit'].nunique()} subreddits")
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')  
    plt.title(f"Cluster {cluster_num}")
    plt.show()

In [ ]:
ari = adjusted_rand_score(data["subreddit"], data["cluster"])
nmi = normalized_mutual_info_score(data["subreddit"], data["cluster"])
ami = adjusted_mutual_info_score(data["subreddit"], data["cluster"])

print("ARI:", ari)
print("NMI:", nmi)
print("AMI:", ami)

In [ ]:
#plot for each cluster the length of comments (boxplot)
plt.figure(figsize=(10, 6))
for i in range():
    cluster_data = data[data['cluster'] == i]
    sns.boxplot(x=cluster_data['cluster'], y=cluster_data['body'].apply(lambda x: len(x.split())))
plt.xlabel('Cluster')

#print for each cluster average length of comments, max, min 25th percentile, 75th percentile
for i in range():
    cluster_data = data[data['cluster'] == i]
    comment_lengths = cluster_data['body'].apply(lambda x: len(x.split()))
    print(f"Cluster {i}:")
    print(f"  Min length: {comment_lengths.min()} words")
    print(f"  25th percentile: {comment_lengths.quantile(0.25)} words")
    print(f"  Average length: {comment_lengths.mean():.2f} words")
    print(f"  75th percentile: {comment_lengths.quantile(0.75)} words")
    print(f"  Max length: {comment_lengths.max()} words")
